<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Transformation — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [ ]:
# Import the numerical, interpolation and image I/O tools used throughout the lab.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import map_coordinates

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths


In [ ]:
# Resolve the lab root before loading images or writing outputs.
def find_lab_root(start: Path) -> Path:
    """Locate the nearest valid Image Transformation lab directory.
    
    Parameters
    ----------
    start : Path
        Directory where the upward search begins.
    
    Returns
    -------
    Path
        First ancestor containing both data/ and notebooks/.
    
    Notes
    -----
    This avoids hard-coded absolute paths while protecting against accidental
    selection of an unrelated parent directory.
    """
    start = start.resolve()

    for candidate in [start, *start.parents]:
        # Accept only a parent that contains both the data source and notebook context.
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the Image_Transformation lab root."
    )


LAB_DIR = find_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "ascent": DATA_DIR / "ascentB.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "einstein": DATA_DIR / "einstein.png",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
    "peppers": DATA_DIR / "peppers.png",
}

missing = [path.name for path in IMAGE_FILES.values() if not path.exists()]
assert not missing, f"Missing input files: {missing}"

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images found  :", len(IMAGE_FILES))

## 2. Load and Inspect the Reference Images


In [ ]:
# Load RGB and grayscale views once so all transformations share the same inputs.
images_rgb = {
    name: np.asarray(Image.open(path).convert("RGB"))
    for name, path in IMAGE_FILES.items()
}

images_gray = {
    name: np.asarray(Image.open(path).convert("L"))
    for name, path in IMAGE_FILES.items()
}

for name in IMAGE_FILES:
    rgb = images_rgb[name]
    gray = images_gray[name]

    print(
        f"{name:9s} | "
        f"RGB={str(rgb.shape):16s} "
        f"gray={str(gray.shape):12s} "
        f"dtype={gray.dtype} "
        f"range=[{gray.min()}, {gray.max()}]"
    )

In [ ]:
# Verify all reference images before comparing transformation behavior.
fig, axes = plt.subplots(1, len(images_rgb), figsize=(16, 4))

for ax, (name, image) in zip(axes, images_rgb.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Intensity Transformation Model


In [ ]:
# Establish identity as the baseline every transformation should be compared against.
ascent = images_gray["ascent"]

identity = ascent.copy()

assert np.array_equal(identity, ascent)

print("Identity transformation preserves every pixel:", np.array_equal(identity, ascent))

## 4. Image Negative


In [ ]:
# Use exact 8-bit inversion as the simplest pointwise intensity transform.
negative = 255 - ascent

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(negative, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Negative")

axes[2].plot(np.arange(256), 255 - np.arange(256))
axes[2].set_title("Transformation: s = 255 - r")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].set_xlim(0, 255)
axes[2].set_ylim(0, 255)
axes[2].grid(alpha=0.25)

for ax in axes[:2]:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_negative.png", dpi=300, bbox_inches="tight")
plt.show()


> **Output comment.** The negative transformation reverses the intensity ordering: bright regions become dark and dark regions become bright. The geometry is unchanged, so any visual difference is purely radiometric. This makes the operation useful for emphasizing structures whose contrast is easier to perceive after inversion, especially in images dominated by dark backgrounds.


## 5. Brightness and Contrast


In [ ]:
# Perform brightness/contrast mapping in float, then clip only at storage boundaries.
def linear_intensity_transform(
    image: np.ndarray,
    gain: float = 1.0,
    offset: float = 0.0,
) -> np.ndarray:
    """Apply the affine intensity model s = gain*r + offset.
    
    Parameters
    ----------
    image : ndarray
        uint8 input image.
    gain : float
        Multiplicative contrast factor.
    offset : float
        Additive brightness shift.
    
    Returns
    -------
    ndarray
        uint8 transformed image.
    
    Notes
    -----
    Arithmetic is performed in float because intermediate values may leave the
    storage range. Clipping is intentionally delayed until the final cast.
    """
    transformed = gain * image.astype(np.float32) + offset
    return np.clip(transformed, 0, 255).astype(np.uint8)


# ±50 gives a visible brightness shift while preserving substantial unsaturated content.
brighter = linear_intensity_transform(ascent, gain=1.0, offset=50)
darker = linear_intensity_transform(ascent, gain=1.0, offset=-50)
# Gain 1.5 expands contrast; the negative offset recenters mid-tones after scaling.
higher_contrast = linear_intensity_transform(ascent, gain=1.5, offset=-64)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

examples = [
    ("Original", ascent),
    ("Brightness +50", brighter),
    ("Brightness -50", darker),
    ("Higher contrast", higher_contrast),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_brightness_contrast.png", dpi=300, bbox_inches="tight")
plt.show()


> **Output comment.** Brightness changes shift the intensity distribution, while contrast changes scale differences around a reference level. Clipping at the valid image range is the key numerical effect to watch: aggressive parameters can collapse distinct input values to 0 or 255 and permanently remove information.


## 6. Contrast Stretching


In [ ]:
# Stretch only the occupied intensity range while handling constant images safely.
def contrast_stretch(image: np.ndarray) -> np.ndarray:
    """Expand the observed intensity interval to the full 8-bit range.
    
    Parameters
    ----------
    image : ndarray
        Grayscale uint8 image.
    
    Returns
    -------
    ndarray
        Contrast-stretched uint8 image.
    
    Notes
    -----
    The mapping preserves intensity order. A constant image has no interval to
    stretch and is returned as a stable all-zero result.
    """
    image_f = image.astype(np.float32)
    r_min = image_f.min()
    r_max = image_f.max()

    # A constant image has no contrast interval to expand.
    if r_max == r_min:
        return np.zeros_like(image)

    stretched = (image_f - r_min) / (r_max - r_min)
    stretched *= 255.0

    return np.clip(stretched, 0, 255).astype(np.uint8)


# Compress intensities deliberately so contrast stretching has a clear measurable effect.
low_contrast = 90 + (ascent.astype(np.float32) / 255.0) * 75
low_contrast = np.clip(low_contrast, 0, 255).astype(np.uint8)

stretched = contrast_stretch(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before stretching")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(stretched, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("Contrast stretched")
axes[1, 0].axis("off")

axes[1, 1].hist(stretched.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After stretching")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_contrast_stretching.png", dpi=300, bbox_inches="tight")
plt.show()

print("Before:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After :", int(stretched.min()), "to", int(stretched.max()))


> **Output comment.** Contrast stretching expands the occupied input range over a larger output range. It is most beneficial when the original histogram is compressed, but the improvement is a remapping of existing intensities rather than recovery of missing detail.


## 7. Logarithmic Transformation


In [ ]:
# Scale the logarithmic mapping so the 8-bit output range remains valid.
def log_transform(image: np.ndarray) -> np.ndarray:
    """Apply a logarithmic 8-bit intensity mapping.
    
    Parameters
    ----------
    image : ndarray
        Grayscale uint8 image.
    
    Returns
    -------
    ndarray
        Log-transformed uint8 image.
    
    Notes
    -----
    The scale constant is chosen so input 255 maps back to 255, enabling direct
    comparison with other 8-bit transformations.
    """
    image_f = image.astype(np.float32)

    c = 255.0 / np.log1p(255.0)
    transformed = c * np.log1p(image_f)

    return np.clip(transformed, 0, 255).astype(np.uint8)


logged = log_transform(ascent)

r = np.arange(256, dtype=np.float32)
log_curve = (255.0 / np.log1p(255.0)) * np.log1p(r)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(logged, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Log transform")
axes[1].axis("off")

axes[2].plot(r, log_curve)
axes[2].set_title("Log transformation curve")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_log_transform.png", dpi=300, bbox_inches="tight")
plt.show()


> **Output comment.** The logarithmic mapping expands low intensities and compresses high intensities. The resulting image therefore reveals structure in darker regions while reducing separation among already bright values. This is appropriate when useful information is concentrated near the low end of the dynamic range.


## 8. Gamma / Power-Law Transformation


In [ ]:
# Normalize intensities before applying the power law to keep gamma interpretable.
def gamma_transform(image: np.ndarray, gamma: float) -> np.ndarray:
    """Apply a normalized power-law intensity transformation.
    
    Parameters
    ----------
    image : ndarray
        uint8 input image.
    gamma : float
        Positive exponent controlling tonal emphasis.
    
    Returns
    -------
    ndarray
        Gamma-transformed uint8 image.
    
    Notes
    -----
    Input is normalized to [0,1], so gamma < 1 brightens darker values and
    gamma > 1 suppresses them. Non-positive gamma is rejected.
    """
    # Positive gamma preserves the standard monotonic power-law mapping.
    if gamma <= 0:
        raise ValueError("gamma must be strictly positive.")

    normalized = image.astype(np.float32) / 255.0
    transformed = normalized ** gamma

    return np.clip(transformed * 255.0, 0, 255).astype(np.uint8)


# Span strong brightening, identity, and strong darkening in one controlled sweep.
gammas = [0.4, 0.7, 1.0, 1.5, 2.2]

fig, axes = plt.subplots(1, len(gammas), figsize=(16, 3.6))

for ax, gamma in zip(axes, gammas):
    transformed = gamma_transform(ascent, gamma)
    ax.imshow(transformed, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"γ = {gamma}")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_gamma_examples.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot transfer curves so gamma behavior is visible independently of one image.
r = np.linspace(0, 1, 256)

fig, ax = plt.subplots(figsize=(6, 4.5))

for gamma in gammas:
    ax.plot(r, r ** gamma, label=f"γ={gamma}")

ax.plot(r, r, linestyle="--", label="identity")
ax.set_title("Gamma / Power-Law Curves")
ax.set_xlabel("Normalized input r")
ax.set_ylabel("Normalized output s")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_gamma_curves.png", dpi=300, bbox_inches="tight")
plt.show()


> **Output comment.** Gamma provides continuous control over the tonal response. Values below 1 brighten darker intensities, whereas values above 1 suppress them. The gamma curves make clear that the same input image can be enhanced differently depending on whether the application needs shadow detail or highlight compression.


## 9. Histogram Equalization


In [ ]:
def histogram_equalize(image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Equalize a grayscale image using its cumulative histogram.
    
    Parameters
    ----------
    image : ndarray
        8-bit grayscale image.
    
    Returns
    -------
    equalized : ndarray
        Equalized uint8 image.
    mapping : ndarray
        256-entry monotonic lookup table.
    
    Notes
    -----
    The mapping is computed once from the CDF and applied by indexing, avoiding
    per-pixel Python loops.
    """
    histogram = np.bincount(image.ravel(), minlength=256)

    probability = histogram / image.size
    cdf = np.cumsum(probability)

    # Use the normalized CDF as a monotonic intensity mapping.
    mapping = np.round(255 * cdf).astype(np.uint8)
    equalized = mapping[image]

    return equalized, mapping


equalized, equalization_map = histogram_equalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Before equalization")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Original histogram")

axes[1, 0].imshow(equalized, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After equalization")
axes[1, 0].axis("off")

axes[1, 1].hist(equalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("Equalized histogram")

for ax in [axes[0, 1], axes[1, 1]]:
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Pixel count")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_histogram_equalization.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Inspect the equalization map directly to verify monotonic intensity ordering.
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(np.arange(256), equalization_map)
ax.set_title("Histogram Equalization Mapping")
ax.set_xlabel("Input intensity")
ax.set_ylabel("Mapped intensity")
ax.set_xlim(0, 255)
ax.set_ylim(0, 255)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_equalization_mapping.png", dpi=300, bbox_inches="tight")
plt.show()


> **Output comment.** Histogram equalization redistributes intensity levels using the cumulative distribution rather than applying one fixed global slope. It can improve global contrast when intensities occupy a restricted range, but the mapping is data-dependent and may also over-emphasize noise or alter the visual balance of already well-exposed regions.


## 10. Compare the Fundamental Intensity Transformations


In [ ]:
# Compare all pointwise transforms on the same source image and display scale.
comparison = [
    ("Original", ascent),
    ("Negative", negative),
    ("Brighter", brighter),
    ("Contrast stretch", contrast_stretch(ascent)),
    ("Log", logged),
    ("Gamma 0.5", gamma_transform(ascent, 0.5)),
    ("Gamma 2.0", gamma_transform(ascent, 2.0)),
    ("Hist. equalized", histogram_equalize(ascent)[0]),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, (title, image) in zip(axes.ravel(), comparison):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "10_intensity_transform_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Geometric Transformation Model

## 12. Homogeneous Coordinates


In [ ]:
# Validate homogeneous transforms first on individual coordinates.
def apply_to_point(matrix: np.ndarray, x: float, y: float) -> np.ndarray:
    """Apply one homogeneous 2-D transform to a Cartesian point.
    
    Parameters
    ----------
    matrix : ndarray
        3x3 homogeneous transform.
    x, y : float
        Cartesian input coordinates.
    
    Returns
    -------
    ndarray
        Dehomogenized output coordinates with shape (2,).
    """
    point = np.array([x, y, 1.0], dtype=np.float64)
    transformed = matrix @ point
    return transformed[:2] / transformed[2]


identity_matrix = np.eye(3)

print("Identity matrix:")
print(identity_matrix)
print("Point (10, 20) ->", apply_to_point(identity_matrix, 10, 20))

## 13. Fundamental Geometric Transformation Matrices


In [ ]:
# Represent all geometric transforms in one homogeneous-coordinate convention.
def translation_matrix(tx: float, ty: float) -> np.ndarray:
    """Construct a homogeneous 2-D translation.
    
    Parameters
    ----------
    tx, ty : float
        Horizontal and vertical displacement.
    
    Returns
    -------
    ndarray
        3x3 translation matrix.
    """
    return np.array(
        [[1.0, 0.0, tx],
         [0.0, 1.0, ty],
         [0.0, 0.0, 1.0]]
    )


def scaling_matrix(sx: float, sy: float) -> np.ndarray:
    """Construct independent homogeneous x/y scaling.
    
    Parameters
    ----------
    sx, sy : float
        Horizontal and vertical scale factors.
    
    Returns
    -------
    ndarray
        3x3 scaling matrix.
    """
    return np.array(
        [[sx, 0.0, 0.0],
         [0.0, sy, 0.0],
         [0.0, 0.0, 1.0]]
    )


def rotation_matrix(angle_degrees: float) -> np.ndarray:
    """Construct a counter-clockwise homogeneous rotation.
    
    Parameters
    ----------
    angle_degrees : float
        Rotation angle in degrees.
    
    Returns
    -------
    ndarray
        3x3 rotation matrix about the coordinate origin.
    """
    angle = np.deg2rad(angle_degrees)
    c = np.cos(angle)
    s = np.sin(angle)

    return np.array(
        [[c, -s, 0.0],
         [s,  c, 0.0],
         [0.0, 0.0, 1.0]]
    )


def shear_matrix(kx: float = 0.0, ky: float = 0.0) -> np.ndarray:
    """Construct a homogeneous 2-D shear transform.
    
    Parameters
    ----------
    kx, ky : float
        Horizontal and vertical shear coefficients.
    
    Returns
    -------
    ndarray
        3x3 shear matrix.
    """
    return np.array(
        [[1.0, kx, 0.0],
         [ky, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )


def horizontal_reflection_matrix() -> np.ndarray:
    """Construct reflection across the y-axis.
    
    Returns
    -------
    ndarray
        3x3 homogeneous reflection matrix.
    
    Notes
    -----
    A later translation is required when reflecting image coordinates so the result
    remains within the positive image frame.
    """
    return np.array(
        [[-1.0, 0.0, 0.0],
         [0.0, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )

## 14. Origin-Centered vs Centered Geometry


In [ ]:
# Conjugate a transform with translations so its pivot is the image center.
def around_center(
    matrix: np.ndarray,
    image_shape: tuple[int, ...],
) -> np.ndarray:
    """Move an existing transform so the image center is its pivot.
    
    Parameters
    ----------
    matrix : ndarray
        3x3 transform defined about the origin.
    image_shape : tuple
        Image shape used to compute the geometric center.
    
    Returns
    -------
    ndarray
        Center-conjugated transform T(c) @ matrix @ T(-c).
    
    Notes
    -----
    The rightmost operation acts first under the column-vector convention used by
    the notebook.
    """
    height, width = image_shape[:2]

    cx = (width - 1) / 2.0
    cy = (height - 1) / 2.0

    to_origin = translation_matrix(-cx, -cy)
    back = translation_matrix(cx, cy)

    return back @ matrix @ to_origin


> **Output comment.** Rotation or scaling about the image origin produces a visible translation of the content because the origin is at a corner. Re-centering the transformation around the image center preserves the intended visual pivot. This demonstrates that the same rotation matrix can produce very different results depending on the coordinate frame in which it is applied.


## 15. Forward Mapping vs Inverse Mapping


In [ ]:
def warp_affine(
    image: np.ndarray,
    forward_matrix: np.ndarray,
    output_shape: tuple[int, int] | None = None,
    interpolation_order: int = 1,
    fill_value: float = 0.0,
) -> np.ndarray:
    """Resample an image from a forward affine model using inverse mapping.
    
    Parameters
    ----------
    image : ndarray
        2-D grayscale or 3-D color image.
    forward_matrix : ndarray
        3x3 source-to-destination affine transform.
    output_shape : tuple[int, int] | None
        Requested output height and width; defaults to input spatial dimensions.
    interpolation_order : int
        SciPy interpolation order: 0 nearest, 1 bilinear, 3 cubic in this lab.
    fill_value : float
        Value assigned when inverse-mapped coordinates fall outside the source.
    
    Returns
    -------
    ndarray
        uint8 warped image.
    
    Notes
    -----
    Inverse mapping visits every destination pixel and therefore avoids holes
    created by forward splatting. Color channels share identical geometric
    coordinates and are resampled independently.
    """

    # Preserve input dimensions by default so geometry changes are easy to compare.
    if output_shape is None:
        output_shape = image.shape[:2]

    out_h, out_w = output_shape

    # Enumerate destination pixels so inverse mapping produces dense output.
    yy, xx = np.indices((out_h, out_w), dtype=np.float64)
    homogeneous_output = np.stack(
        [xx.ravel(), yy.ravel(), np.ones(xx.size)],
        axis=0,
    )

    # Map each destination coordinate back to its source location.
    inverse_matrix = np.linalg.inv(forward_matrix)
    homogeneous_input = inverse_matrix @ homogeneous_output

    x_in = homogeneous_input[0]
    y_in = homogeneous_input[1]

    coordinates = np.vstack([y_in, x_in])

    # Grayscale needs one interpolation pass; color is resampled channel-wise.
    if image.ndim == 2:
        warped = map_coordinates(
            image.astype(np.float32),
            coordinates,
            order=interpolation_order,
            mode="constant",
            cval=fill_value,
        ).reshape(out_h, out_w)

    # Apply the same geometric coordinates to every color channel.
    elif image.ndim == 3:
        channels = []
        for channel_index in range(image.shape[2]):
            sampled = map_coordinates(
                image[..., channel_index].astype(np.float32),
                coordinates,
                order=interpolation_order,
                mode="constant",
                cval=fill_value,
            ).reshape(out_h, out_w)
            channels.append(sampled)

        warped = np.stack(channels, axis=-1)

    else:
        raise ValueError("Expected a 2-D grayscale or 3-D color image.")

    return np.clip(warped, 0, 255).astype(np.uint8)


> **Output comment.** Forward mapping can leave unassigned pixels because transformed source coordinates do not necessarily land on every output location. Inverse mapping instead asks where each output pixel originated and is therefore the standard strategy for dense image warping. Interpolation then estimates the source value at non-integer coordinates.


## 16. Translation


In [ ]:
# Apply translation through the shared affine warp rather than special-case indexing.
einstein = images_gray["einstein"]

# Use a large visible translation while keeping much of the image inside the canvas.
T = translation_matrix(tx=70, ty=35)
translated = warp_affine(
    einstein,
    T,
    # Order 0 preserves nearest-neighbor behavior for integer-like translation/reflection.
interpolation_order=0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(translated, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translation: tx=70, ty=35")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_translation.png", dpi=300, bbox_inches="tight")
plt.show()

## 17. Rotation


In [ ]:
# Contrast rotation about the image origin with rotation about its center.
# Thirty degrees is large enough to expose pivot choice without making content unrecognizable.
R_origin = rotation_matrix(30)
R_center = around_center(rotation_matrix(30), einstein.shape)

rotated_origin = warp_affine(einstein, R_origin, interpolation_order=1)
rotated_center = warp_affine(einstein, R_center, interpolation_order=1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(rotated_origin, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Rotation about origin")

axes[2].imshow(rotated_center, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotation about image center")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_rotation_origin_center.png", dpi=300, bbox_inches="tight")
plt.show()

## 18. Scaling and Resizing


In [ ]:
# Compare uniform and anisotropic scaling around the same geometric pivot.
S_uniform = around_center(
    # 0.65 demonstrates uniform shrinkage while retaining enough structure for comparison.
scaling_matrix(0.65, 0.65),
    einstein.shape,
)

S_nonuniform = around_center(
    # Opposing x/y scales deliberately expose anisotropic shape distortion.
scaling_matrix(1.35, 0.65),
    einstein.shape,
)

scaled_uniform = warp_affine(
    einstein,
    S_uniform,
    # Order 1 uses bilinear interpolation as the default smooth geometric baseline.
interpolation_order=1,
)

scaled_nonuniform = warp_affine(
    einstein,
    S_nonuniform,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(scaled_uniform, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Uniform scale")

axes[2].imshow(scaled_nonuniform, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Non-uniform scale")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "13_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

## 19. Interpolation Comparison


In [ ]:
peppers_rgb = images_rgb["peppers"]

scale_for_interpolation = around_center(
    # Upscale by 1.7 so interpolation differences become visible on the same crop.
scaling_matrix(1.7, 1.7),
    peppers_rgb.shape,
)

nearest = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=0,
)
bilinear = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=1,
)
bicubic = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    # Order 3 provides a smoother bicubic-style comparison at higher computational cost.
interpolation_order=3,
)

# Compare interpolation on the same spatial crop.
h, w = peppers_rgb.shape[:2]
crop = (
    slice(h // 3, 2 * h // 3),
    slice(w // 3, 2 * w // 3),
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(peppers_rgb)
axes[0].set_title("Original")

axes[1].imshow(nearest[crop])
axes[1].set_title("Nearest")

axes[2].imshow(bilinear[crop])
axes[2].set_title("Bilinear")

axes[3].imshow(bicubic[crop])
axes[3].set_title("Bicubic")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "14_interpolation_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


> **Output comment.** Nearest-neighbor interpolation preserves original sample values but can create blocky edges. Bilinear interpolation smooths transitions by combining four neighbors, while bicubic interpolation uses a larger neighborhood and usually produces smoother gradients. The comparison shows that interpolation choice is a quality-versus-cost decision rather than a change in the underlying geometry.


## 20. Reflection / Flipping


In [ ]:
# Compose reflection with translation so the result remains inside the image frame.
height, width = einstein.shape

F = translation_matrix(width - 1, 0) @ horizontal_reflection_matrix()
reflected = warp_affine(einstein, F, interpolation_order=0)

reflected_numpy = einstein[:, ::-1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(reflected, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Homogeneous transform")

axes[2].imshow(reflected_numpy, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("NumPy slicing")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "15_reflection.png", dpi=300, bbox_inches="tight")
plt.show()

print("Two reflection methods identical:", np.array_equal(reflected, reflected_numpy))

## 21. Shear


In [ ]:
# Center the shear to separate shape distortion from unintended translation.
shear_centered = around_center(
    # kx=0.35 creates an obvious horizontal shear without collapsing image geometry.
shear_matrix(kx=0.35, ky=0.0),
    einstein.shape,
)

sheared = warp_affine(
    einstein,
    shear_centered,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(sheared, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Horizontal shear")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "16_shear.png", dpi=300, bbox_inches="tight")
plt.show()

## 22. Composition of Transformations


In [ ]:
# Reverse transform order explicitly to demonstrate non-commutativity.
# Pair a visible translation with rotation to demonstrate non-commutative composition.
translate = translation_matrix(70, 20)
# A 25-degree centered rotation keeps the composition visually interpretable.
rotate = around_center(rotation_matrix(25), einstein.shape)

translate_then_rotate = rotate @ translate
rotate_then_translate = translate @ rotate

image_a = warp_affine(
    einstein,
    translate_then_rotate,
    interpolation_order=1,
)

image_b = warp_affine(
    einstein,
    rotate_then_translate,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(image_a, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translate → Rotate")

axes[2].imshow(image_b, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotate → Translate")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "17_transformation_order.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


> **Output comment.** Transformation order matters because affine matrices generally do not commute. Translating and then rotating is not equivalent to rotating and then translating. Using homogeneous matrices makes the order explicit and allows the complete sequence to be represented by one composite matrix.


## 23. General Affine Transformation


In [ ]:
# Compose several affine operators into one matrix before resampling once.
affine = (
    translation_matrix(35, -10)
    @ around_center(rotation_matrix(-18), einstein.shape)
    @ around_center(shear_matrix(kx=0.18), einstein.shape)
    @ around_center(scaling_matrix(0.88, 1.08), einstein.shape)
)

affine_result = warp_affine(
    images_rgb["einstein"],
    affine,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(images_rgb["einstein"])
axes[0].set_title("Original")

axes[1].imshow(affine_result)
axes[1].set_title("Combined affine transform")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "18_affine_transform.png", dpi=300, bbox_inches="tight")
plt.show()


> **Output comment.** An affine transform can combine translation, rotation, scaling, shear, and reflection while preserving straight lines and parallelism. The experiment demonstrates the practical advantage of the matrix formulation: complex geometric changes can be expressed, composed, and validated within one consistent coordinate model.


## 24. Resizing to a New Array Shape


In [ ]:
# Compare interpolation methods at the same target resolution.
source = Image.fromarray(images_rgb["peppers"])

original_width, original_height = source.size

# Downsample by exactly 2x so interpolation methods share one controlled target size.
new_width = original_width // 2
new_height = original_height // 2

resized_nearest = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.NEAREST,
    )
)

resized_bilinear = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BILINEAR,
    )
)

resized_bicubic = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BICUBIC,
    )
)

print("Original shape :", images_rgb["peppers"].shape)
print("Resized shape  :", resized_bilinear.shape)
print(
    "Aspect ratios  :",
    round(original_width / original_height, 4),
    "→",
    round(new_width / new_height, 4),
)

## 25. Validation Checks


In [ ]:
assert np.array_equal(identity, ascent)
assert negative.dtype == np.uint8
assert negative.min() >= 0 and negative.max() <= 255
assert brighter.dtype == np.uint8

# Gamma = 1 is the identity mapping.
gamma_identity = gamma_transform(ascent, 1.0)
assert np.array_equal(gamma_identity, ascent)

# Histogram equalization must preserve intensity ordering.
assert np.all(np.diff(equalization_map.astype(np.int16)) >= 0)

assert translated.shape == einstein.shape
assert rotated_center.shape == einstein.shape
assert affine_result.shape == images_rgb["einstein"].shape

# Validate geometry with a point whose translated location is known exactly.
known_point = apply_to_point(translation_matrix(12, -5), 10, 20)
assert np.allclose(known_point, [22, 15])

# Reflection must match direct left-right reversal.
assert np.array_equal(reflected, reflected_numpy)

# The selected affine transforms must remain invertible.
assert not np.isclose(np.linalg.det(translation_matrix(10, 20)), 0)
assert not np.isclose(np.linalg.det(rotation_matrix(30)), 0)
assert not np.isclose(np.linalg.det(scaling_matrix(0.5, 1.5)), 0)

print("All image-transformation validation checks passed.")


> **Output comment.** The validation stage checks shapes, finite values, intensity ranges, and geometric consistency of representative transformed coordinates. Passing these checks confirms that the transformations are numerically well-formed and that the generated figures correspond to the intended operators.


## Final Analysis & Interpretation

### Main findings

- Point transformations and image warps are most consistently expressed with homogeneous matrices, which allow translation, rotation, scaling, shear, reflection, and composition within one coordinate model.
- Intensity transforms affect radiometry rather than geometry: linear mappings control brightness/contrast, logarithmic mappings emphasize darker values, gamma controls tonal response continuously, and histogram equalization adapts the mapping to the image distribution.
- Rotation or scaling around the image origin produces different visual behavior from the same transform applied around the image center, demonstrating that the coordinate frame is part of the transformation definition.
- Inverse mapping provides dense image warping, while interpolation determines how non-integer source coordinates are sampled. Nearest, bilinear, and cubic interpolation represent different quality-versus-cost compromises.
- Transformation order is non-commutative: translating then rotating is generally not equivalent to rotating then translating.
- The final validation checks confirm consistent shapes, finite values, valid ranges, and representative geometric behavior.

### Engineering interpretation

Homogeneous coordinates are the key abstraction of this lab because they make complex transformations explicit, composable, and testable. Once a forward geometric model is defined, inverse mapping separates geometry from resampling: the matrix determines where a destination pixel comes from, while interpolation determines how its intensity is estimated.

### Limitations

Keeping the output canvas equal to the input size can crop transformed content. Interpolation introduces resampling error, and repeated independent warps can accumulate blur; composing transforms first and resampling once is preferable when possible.

### Final conclusion

The notebook provides a complete transformation workflow linking radiometric mappings, affine geometry, coordinate-frame reasoning, inverse warping, interpolation, transform composition, resizing, and numerical validation.
